In [1]:
# ==========================================================================
# STEP 1 — RECONNAISSANCE  (CPU only, ~1 min, no GPU, no pip installs)
# Reads only zarr.json metadata files. Zero heavy dependencies.
# ==========================================================================
import json, os, re, zipfile, sys
from pathlib import Path
from collections import defaultdict, Counter

INPUT = Path("/kaggle/input")

# ---------- 1. locate competition data ----------
def find_comp_dir():
    named = [
        INPUT / "biohub-cell-tracking-during-development",
        INPUT / "competitions" / "biohub-cell-tracking-during-development",
    ]
    for c in named:
        if c.exists():
            return c
    for p in sorted(INPUT.glob("*")):
        if (p / "train").is_dir() and (p / "test").is_dir():
            return p
        for q in sorted(p.glob("*")):
            if (q / "train").is_dir() and (q / "test").is_dir():
                return q
    return None

COMP = find_comp_dir()
print("=" * 78)
print("COMP_DIR:", COMP)
if COMP is None:
    print("!! competition data NOT attached. Attach it and re-run.")
    sys.exit(0)
TRAIN, TEST = COMP / "train", COMP / "test"
print("train exists:", TRAIN.exists(), "| test exists:", TEST.exists())

# ---------- 2. inventory samples ----------
def stems(d, suffix):
    if not d.exists():
        return []
    return sorted(p.name[: -len(suffix)] for p in d.iterdir() if p.name.endswith(suffix))

train_zarr = stems(TRAIN, ".zarr")
train_geff = stems(TRAIN, ".geff")
test_zarr  = stems(TEST,  ".zarr")

print(f"\nTRAIN: {len(train_zarr)} .zarr | {len(train_geff)} .geff")
print(f"TEST : {len(test_zarr)} .zarr")
print("unpaired train zarr (no geff):", sorted(set(train_zarr) - set(train_geff))[:5])

def embryo(stem):  # "44b6_0049_0438_1330_1273" -> "44b6"
    return stem.split("_")[0]

tr_emb = Counter(embryo(s) for s in train_zarr)
te_emb = Counter(embryo(s) for s in test_zarr)
print(f"\nTRAIN embryos ({len(tr_emb)}):", dict(sorted(tr_emb.items())))
print(f"TEST  embryos ({len(te_emb)}):", dict(sorted(te_emb.items())))
print("overlap (must be empty):", sorted(set(tr_emb) & set(te_emb)))

# ---------- 3. read zarr/geff metadata (pure JSON, no zarr lib) ----------
def jload(p):
    try:
        return json.loads(p.read_text())
    except Exception:
        return {}

def img_meta(p):
    m = jload(p / "0" / "zarr.json")
    return m.get("shape"), m.get("data_type", m.get("dtype"))

def deep_get_est(meta):
    """estimated_number_of_nodes can be nested under attributes/geff."""
    stack = [meta]
    while stack:
        cur = stack.pop()
        if isinstance(cur, dict):
            if "estimated_number_of_nodes" in cur:
                return cur["estimated_number_of_nodes"]
            stack.extend(cur.values())
        elif isinstance(cur, list):
            stack.extend(cur)
    return None

def geff_meta(p):
    root = jload(p / "zarr.json")
    est = deep_get_est(root)
    nsh = jload(p / "nodes" / "ids" / "zarr.json").get("shape")
    esh = jload(p / "edges" / "ids" / "zarr.json").get("shape")
    props = p / "nodes" / "props"
    plist = sorted(x.name for x in props.iterdir()) if props.exists() else []
    return est, nsh, esh, plist

print("\n" + "=" * 78)
print("PER-SAMPLE TRAIN SUMMARY")
print("=" * 78)
print(f"{'stem':<30}{'T,Z,Y,X':<22}{'gt_nodes':>9}{'gt_edges':>9}{'est_true':>10}{'dens%':>7}")

rows = []
for s in train_zarr:
    shape, dtype = img_meta(TRAIN / f"{s}.zarr")
    est = nsh = esh = None
    plist = []
    if (TRAIN / f"{s}.geff").exists():
        est, nsh, esh, plist = geff_meta(TRAIN / f"{s}.geff")
    n_nodes = nsh[0] if nsh else 0
    n_edges = esh[0] if esh else 0
    dens = 100.0 * n_nodes / est if est else float("nan")
    rows.append(dict(stem=s, shape=shape, dtype=dtype, n_nodes=n_nodes,
                     n_edges=n_edges, est=est, dens=dens, props=plist))
    print(f"{s:<30}{str(shape):<22}{n_nodes:>9}{n_edges:>9}{str(est):>10}{dens:>7.1f}")

if rows:
    tot_n = sum(r["n_nodes"] for r in rows)
    tot_e = sum(r["n_edges"] for r in rows)
    tot_est = sum(r["est"] or 0 for r in rows)
    print(f"\nTOTAL gt_nodes={tot_n:,}  gt_edges={tot_e:,}  est_true_nodes={tot_est:,}")
    print(f"GLOBAL ANNOTATION DENSITY: {100.0*tot_n/max(tot_est,1):.2f}%")
    print("node prop keys:", rows[0]["props"])
    print("image dtypes seen:", sorted({str(r['dtype']) for r in rows}))
    print("image shapes seen:", sorted({str(r['shape']) for r in rows})[:8])

print("\nTEST image shapes:")
for s in test_zarr:
    print("  ", s, img_meta(TEST / f"{s}.zarr"))

# ---------- 4. locate the OFFICIAL scorer inside the artifact pack ----------
print("\n" + "=" * 78)
print("SEARCHING FOR OFFICIAL METRIC CODE (tracking_cellmot)")
print("=" * 78)

skip = {COMP.resolve()}
roots = []
for p in sorted(INPUT.glob("*")):
    if p.resolve() in skip or not p.is_dir():
        continue
    roots.append(p)
print("non-competition input dirs:", [p.name for p in roots])

metrics_src, metrics_where = None, None
tree_listing = []

for root in roots:
    # (a) unzipped repo on disk
    for m in root.rglob("tracking_cellmot/metrics.py"):
        metrics_src, metrics_where = m.read_text(), str(m)
        pkg = m.parent
        tree_listing = sorted(x.name for x in pkg.iterdir())
        break
    if metrics_src:
        break
    # (b) repo.zip
    for z in root.rglob("*.zip"):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                hit = [n for n in names if n.endswith("tracking_cellmot/metrics.py")]
                if hit:
                    metrics_src = zf.read(hit[0]).decode("utf-8", "replace")
                    metrics_where = f"{z}::{hit[0]}"
                    tree_listing = [n for n in names
                                    if "tracking_cellmot/" in n or "scripts/" in n]
                    break
        except Exception:
            pass
    if metrics_src:
        break

if metrics_src is None:
    print("!! metrics.py NOT found. Attach the pilkwang support-pack dataset,")
    print("   or we reimplement the scorer from the published metrics.md spec.")
else:
    print("FOUND AT:", metrics_where)
    print("\npackage / archive contents:")
    for n in tree_listing[:60]:
        print("   ", n)
    print("\n" + "=" * 78)
    print("---------- metrics.py SOURCE ----------")
    print("=" * 78)
    print(metrics_src)

print("\n" + "=" * 78)
print("STEP 1 COMPLETE")
print("=" * 78)

COMP_DIR: /kaggle/input/competitions/biohub-cell-tracking-during-development
train exists: True | test exists: True

TRAIN: 199 .zarr | 199 .geff
TEST : 4 .zarr
unpaired train zarr (no geff): []

TRAIN embryos (2): {'44b6': 71, '6bba': 128}
TEST  embryos (2): {'44b6': 2, '6bba': 2}
overlap (must be empty): ['44b6', '6bba']

PER-SAMPLE TRAIN SUMMARY
stem                          T,Z,Y,X                gt_nodes gt_edges  est_true  dens%
44b6_0113de3b                 [100, 64, 256, 256]          52       50     25755    0.2
44b6_0b24845f                 [100, 64, 256, 256]          51       49     32795    0.2
44b6_0c582fdc                 [100, 64, 256, 256]          71       70     27958    0.3
44b6_0db75fae                 [100, 64, 256, 256]         157      151     15335    1.0
44b6_12dfb391                 [100, 64, 256, 256]         788      773     58672    1.3
44b6_144b256d                 [100, 64, 256, 256]         121      119     65376    0.2
44b6_1574802b                 [10

In [2]:
# ==========================================================================
# STEP 2 — GROUND TRUTH STRUCTURE  (CPU only, ~2 min, no GPU)
# Manual zarr-v3 reader: no dependency on the notebook's wheel install.
# ==========================================================================
import json, sys
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict

COMP = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
if not COMP.exists():
    COMP = Path("/kaggle/input/biohub-cell-tracking-during-development")
TRAIN = COMP / "train"
SCALE = np.array([1.625, 0.40625, 0.40625])   # z, y, x  µm/voxel

# ---------- decompression backends ----------
def _make_zstd_decoder():
    try:
        import zstandard
        d = zstandard.ZstdDecompressor()
        return lambda b: d.decompressobj().decompress(b), "zstandard"
    except Exception:
        pass
    try:
        from numcodecs import Zstd
        z = Zstd()
        return lambda b: z.decode(b), "numcodecs"
    except Exception:
        pass
    try:
        import pyzstd
        return lambda b: pyzstd.decompress(b), "pyzstd"
    except Exception:
        pass
    return None, None

ZDEC, ZNAME = _make_zstd_decoder()
print("zstd backend:", ZNAME)
if ZDEC is None:
    print("!! no zstd backend available - stop and tell me"); sys.exit(0)

def read_v3_array(path: Path) -> np.ndarray:
    """Minimal zarr-v3 reader for the small 1D/2D arrays inside .geff."""
    meta  = json.loads((path / "zarr.json").read_text())
    shape = tuple(int(s) for s in meta["shape"])
    dt    = np.dtype(meta["data_type"])
    chunk = tuple(int(c) for c in meta["chunk_grid"]["configuration"]["chunk_shape"])

    sep = "/"
    cke = meta.get("chunk_key_encoding", {})
    sep = cke.get("configuration", {}).get("separator", "/")
    prefix = "c"

    little = True
    for c in meta.get("codecs", []):
        if c.get("name") == "bytes":
            little = c.get("configuration", {}).get("endian", "little") == "little"
    dt = dt.newbyteorder("<" if little else ">")

    out = np.zeros(shape, dtype=dt)
    grid = [int(np.ceil(shape[i] / chunk[i])) for i in range(len(shape))]
    for idx in np.ndindex(*grid):
        key = path / (prefix + sep + sep.join(str(i) for i in idx))
        sl = tuple(slice(idx[i]*chunk[i], min((idx[i]+1)*chunk[i], shape[i]))
                   for i in range(len(shape)))
        if not key.exists():
            continue
        raw = ZDEC(key.read_bytes())
        arr = np.frombuffer(raw, dtype=dt)
        want = tuple(s.stop - s.start for s in sl)
        if arr.size == int(np.prod(chunk)):
            arr = arr.reshape(chunk)[tuple(slice(0, w) for w in want)]
        else:
            arr = arr.reshape(want)
        out[sl] = arr
    return out

def load_geff(p: Path):
    ids = read_v3_array(p / "nodes" / "ids")
    t   = read_v3_array(p / "nodes" / "props" / "t" / "values")
    z   = read_v3_array(p / "nodes" / "props" / "z" / "values")
    y   = read_v3_array(p / "nodes" / "props" / "y" / "values")
    x   = read_v3_array(p / "nodes" / "props" / "x" / "values")
    e   = read_v3_array(p / "edges" / "ids")
    return ids, t, z, y, x, e

# ---------- sanity check on one sample ----------
stems = sorted(p.name[:-5] for p in TRAIN.iterdir() if p.name.endswith(".geff"))
probe = stems[0]
ids, t, z, y, x, e = load_geff(TRAIN / f"{probe}.geff")
print(f"\nprobe {probe}: nodes={len(ids)} edges={e.shape} "
      f"t[{t.min()},{t.max()}] z[{z.min()},{z.max()}] "
      f"y[{y.min()},{y.max()}] x[{x.min()},{x.max()}]")
print("edge sample:\n", e[:3])

# ---------- full scan ----------
agg = dict(nodes=0, edges=0, div=0, triple=0, tracks=0, merges=0,
           dt_bad=0, dt1=0)
disp, sis, par, tracklen, degout, degin = [], [], [], [], Counter(), Counter()
per_emb = defaultdict(lambda: dict(nodes=0, edges=0, div=0, disp=[], est=0))
rows = []

for s in stems:
    try:
        ids, t, z, y, x, e = load_geff(TRAIN / f"{s}.geff")
    except Exception as ex:
        print("FAIL", s, ex); continue
    emb = s.split("_")[0]
    pos = np.stack([z, y, x], 1).astype(np.float64) * SCALE
    idx = {int(v): i for i, v in enumerate(ids)}

    src = np.array([idx[int(a)] for a in e[:, 0]]) if len(e) else np.array([], int)
    tgt = np.array([idx[int(b)] for b in e[:, 1]]) if len(e) else np.array([], int)

    if len(e):
        dt = t[tgt].astype(int) - t[src].astype(int)
        agg["dt1"] += int((dt == 1).sum()); agg["dt_bad"] += int((dt != 1).sum())
        d = np.linalg.norm(pos[tgt] - pos[src], axis=1)
        disp.append(d); per_emb[emb]["disp"].append(d)

    co, ci = Counter(src.tolist()), Counter(tgt.tolist())
    for v in co.values(): degout[v] += 1
    for v in ci.values(): degin[v]  += 1
    ndiv = sum(1 for v in co.values() if v == 2)
    ntri = sum(1 for v in co.values() if v >= 3)
    nmerge = sum(1 for v in ci.values() if v >= 2)

    # sister + parent-daughter geometry at divisions
    kids = defaultdict(list)
    for a, b in zip(src, tgt): kids[a].append(b)
    for a, ch in kids.items():
        if len(ch) == 2:
            sis.append(float(np.linalg.norm(pos[ch[0]] - pos[ch[1]])))
            par.append(float(max(np.linalg.norm(pos[ch[0]] - pos[a]),
                                 np.linalg.norm(pos[ch[1]] - pos[a]))))

    # track lengths via union-find on the graph
    parent = list(range(len(ids)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    for a, b in zip(src, tgt):
        ra, rb = find(int(a)), find(int(b))
        if ra != rb: parent[ra] = rb
    comp = Counter(find(i) for i in range(len(ids)))
    tracklen.extend(comp.values())

    est = 0
    try:
        m = json.loads((TRAIN / f"{s}.geff" / "zarr.json").read_text())
        st = [m]
        while st:
            c = st.pop()
            if isinstance(c, dict):
                if "estimated_number_of_nodes" in c:
                    est = int(c["estimated_number_of_nodes"]); break
                st.extend(c.values())
            elif isinstance(c, list): st.extend(c)
    except Exception: pass

    agg["nodes"] += len(ids); agg["edges"] += len(e)
    agg["div"] += ndiv; agg["triple"] += ntri
    agg["tracks"] += len(comp); agg["merges"] += nmerge
    pe = per_emb[emb]
    pe["nodes"] += len(ids); pe["edges"] += len(e); pe["div"] += ndiv; pe["est"] += est
    rows.append((s, emb, len(ids), len(e), ndiv, len(comp), est))

D   = np.concatenate(disp) if disp else np.array([0.0])
SIS = np.array(sis) if sis else np.array([0.0])
PAR = np.array(par) if par else np.array([0.0])
TL  = np.array(tracklen)

def q(a, ps=(50, 75, 90, 95, 99, 99.9, 100)):
    return "  ".join(f"p{p}={np.percentile(a,p):.2f}" for p in ps)

print("\n" + "="*78); print("GLOBAL GT STRUCTURE"); print("="*78)
print(f"nodes={agg['nodes']:,}  edges={agg['edges']:,}  tracks={agg['tracks']:,}")
print(f"divisions (outdeg==2) = {agg['div']:,}   outdeg>=3 = {agg['triple']:,}")
print(f"merges (indeg>=2)     = {agg['merges']:,}   <-- should be 0")
print(f"edges with dt==1: {agg['dt1']:,} | dt!=1: {agg['dt_bad']:,}  <-- should be 0")
print(f"division rate per node   = {100*agg['div']/max(agg['nodes'],1):.3f}%")
print(f"divisions per track      = {agg['div']/max(agg['tracks'],1):.3f}")
print("\noutdegree dist:", dict(sorted(degout.items())))
print("indegree  dist:", dict(sorted(degin.items())))

print("\n--- EDGE DISPLACEMENT (µm, frame-to-frame) ---")
print(" ", q(D)); print(f"  mean={D.mean():.2f}  n={len(D):,}")
for g in (3, 4, 5, 6, 8, 10, 14):
    print(f"   gate {g:>2} µm covers {100*(D<=g).mean():.3f}% of true edges")

print("\n--- DIVISION GEOMETRY (µm) ---")
print("  sister-sister  :", q(SIS), f" n={len(SIS)}")
print("  parent->daughter (max):", q(PAR))
print("  your SAFE_DIV_MAX_UM=4.66 covers "
      f"{100*(PAR<=4.66).mean():.1f}% | SISTER_MAX=8.5 covers {100*(SIS<=8.5).mean():.1f}%")

print("\n--- GT TRACK / COMPONENT LENGTH (nodes) ---")
print(" ", q(TL)); print(f"  mean={TL.mean():.1f}")
for L in (2, 3, 4, 5, 6, 8):
    print(f"   components with <{L} nodes: {100*(TL<L).mean():.2f}% "
          f"({int((TL<L).sum()):,} of {len(TL):,})")

print("\n--- PER EMBRYO ---")
for emb, v in sorted(per_emb.items()):
    dd = np.concatenate(v["disp"]) if v["disp"] else np.array([0.0])
    print(f"  {emb}: nodes={v['nodes']:,} edges={v['edges']:,} div={v['div']:,} "
          f"({100*v['div']/max(v['nodes'],1):.3f}%) est_true={v['est']:,} "
          f"| disp p50={np.percentile(dd,50):.2f} p95={np.percentile(dd,95):.2f} "
          f"p99.9={np.percentile(dd,99.9):.2f}")

print("\n--- 12 SAMPLES WITH MOST DIVISIONS ---")
for r in sorted(rows, key=lambda r: -r[4])[:12]:
    print(f"  {r[0]:<26} emb={r[1]} nodes={r[2]:>5} edges={r[3]:>5} "
          f"div={r[4]:>4} tracks={r[5]:>5} est={r[6]:>6}")
print("\nSTEP 2 COMPLETE")

zstd backend: zstandard

probe 44b6_0113de3b: nodes=52 edges=(50, 2) t[0,75] z[1,63] y[74,230] x[73,253]
edge sample:
 [[11000000075 12000000075]
 [12000000075 13000000075]
 [38000000003 39000000003]]

GLOBAL GT STRUCTURE
nodes=133,318  edges=128,883  tracks=4,435
divisions (outdeg==2) = 151   outdeg>=3 = 0
merges (indeg>=2)     = 0   <-- should be 0
edges with dt==1: 128,883 | dt!=1: 0  <-- should be 0
division rate per node   = 0.113%
divisions per track      = 0.034

outdegree dist: {1: 128581, 2: 151}
indegree  dist: {1: 128883}

--- EDGE DISPLACEMENT (µm, frame-to-frame) ---
  p50=1.82  p75=2.73  p90=4.14  p95=5.34  p99=8.38  p99.9=13.84  p100=60.76
  mean=2.13  n=128,883
   gate  3 µm covers 77.977% of true edges
   gate  4 µm covers 88.764% of true edges
   gate  5 µm covers 93.700% of true edges
   gate  6 µm covers 96.442% of true edges
   gate  8 µm covers 98.687% of true edges
   gate 10 µm covers 99.702% of true edges
   gate 14 µm covers 99.904% of true edges

--- DIVISION

In [3]:
# ==========================================================================
# STEP 1b — ENVIRONMENT / ATTACHMENT INVENTORY  (10 seconds)
# ==========================================================================
import sys, subprocess, importlib.util
from pathlib import Path

print("=" * 78)
print("ATTACHED INPUTS")
print("=" * 78)
INPUT = Path("/kaggle/input")
for p in sorted(INPUT.iterdir()):
    print(f"\n[{p.name}]")
    try:
        kids = sorted(p.iterdir())
    except Exception as e:
        print("   ", e); continue
    for k in kids[:15]:
        tag = "DIR " if k.is_dir() else "FILE"
        print(f"    {tag} {k.name}")
        if k.is_dir() and p.name == "datasets":
            for j in sorted(k.iterdir())[:10]:
                print(f"          - {j.name}")
    if len(kids) > 15:
        print(f"    ... +{len(kids)-15} more")

print("\n" + "=" * 78)
print("INTERNET CHECK")
print("=" * 78)
try:
    r = subprocess.run(["curl", "-sS", "-m", "8", "-o", "/dev/null",
                        "-w", "%{http_code}", "https://pypi.org/simple/"],
                       capture_output=True, text=True, timeout=20)
    print("pypi.org HTTP status:", r.stdout.strip() or r.stderr.strip())
    print("INTERNET:", "ON" if r.stdout.strip().startswith("2") else "OFF / blocked")
except Exception as e:
    print("INTERNET: OFF -", e)

print("\n" + "=" * 78)
print("KEY PACKAGES ALREADY PRESENT")
print("=" * 78)
for m in ["torch", "numpy", "scipy", "zarr", "numcodecs", "zstandard", "pyzstd",
          "polars", "tracksdata", "geff", "networkx", "rustworkx", "skimage",
          "pandas", "sklearn"]:
    spec = importlib.util.find_spec(m)
    v = ""
    if spec:
        try:
            v = " " + str(__import__(m).__version__)
        except Exception:
            v = " (no __version__)"
    print(f"  {m:<14} {'YES' + v if spec else 'no'}")

import torch
print("\nGPU:", torch.cuda.device_count(), "device(s)",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
      if torch.cuda.is_available() else "- CPU only")
print("python:", sys.version.split()[0])
print("\nSTEP 1b COMPLETE")

ATTACHED INPUTS

[competitions]
    DIR  biohub-cell-tracking-during-development

[datasets]
    DIR  pilkwang
          - biohub-deepcenter-unet3d-center-prior-v1
          - biohub-temporal-unet3d-seed314159-v1
          - biohub-tracking-support-pack-50ep-v1
          - pilkwang-public-dataset-for-notebooks-figures

INTERNET CHECK
pypi.org HTTP status: 200
INTERNET: ON

KEY PACKAGES ALREADY PRESENT
  torch          YES 2.10.0+cu128
  numpy          YES 2.0.2
  scipy          YES 1.16.3
  zarr           no
  numcodecs      no
  zstandard      YES 0.25.0
  pyzstd         no
  polars         YES 1.35.2
  tracksdata     no
  geff           no
  networkx       YES 3.6.1
  rustworkx      no
  skimage        YES 0.25.2
  pandas         YES 2.3.3
  sklearn        YES 1.6.1

GPU: 2 device(s) ['Tesla T4', 'Tesla T4']
python: 3.12.13

STEP 1b COMPLETE


In [4]:
# ==========================================================================
# STEP 3 — ACQUIRE OFFICIAL SCORER (read-only, ~30s, no installs)
# ==========================================================================
import subprocess, json, zipfile, sys
from pathlib import Path

# ---------- (a) targeted search of the now-attached pilkwang pack ----------
print("=" * 78); print("PILKWANG PACK CONTENTS"); print("=" * 78)
PK = Path("/kaggle/input/datasets/pilkwang")
found_local = None
if PK.exists():
    for d in sorted(PK.iterdir()):
        print(f"\n[{d.name}]")
        for k in sorted(d.rglob("*"))[:40]:
            if k.is_file():
                print(f"    {k.relative_to(d)}  ({k.stat().st_size/1e6:.1f} MB)")
            elif k.is_dir():
                print(f"    DIR {k.relative_to(d)}/")
    for m in PK.rglob("tracking_cellmot/metrics.py"):
        found_local = m; break
    if not found_local:
        for z in PK.rglob("*.zip"):
            try:
                with zipfile.ZipFile(z) as zf:
                    hit = [n for n in zf.namelist()
                           if n.endswith("tracking_cellmot/metrics.py")]
                    print(f"\n  zip {z.name}: {len(zf.namelist())} entries, "
                          f"metrics.py hit={hit[:1]}")
                    if hit:
                        found_local = f"{z}::{hit[0]}"; break
            except Exception as e:
                print("  zip err", z.name, e)
print("\nlocal metrics.py:", found_local)

# ---------- (b) clone the official repo (internet is ON) ----------
print("\n" + "=" * 78); print("CLONING OFFICIAL REPO"); print("=" * 78)
DST = Path("/kaggle/working/official_repo")
if not DST.exists():
    r = subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/royerlab/kaggle-cell-tracking-competition.git", str(DST)],
        capture_output=True, text=True, timeout=180)
    print("rc:", r.returncode); print(r.stdout[-1500:]); print(r.stderr[-1500:])
else:
    print("already present")

if DST.exists():
    print("\nrepo tree:")
    for p in sorted(DST.rglob("*.py")):
        print("   ", p.relative_to(DST), f"({p.stat().st_size} B)")
    for p in sorted(DST.glob("*.toml")):
        print("   ", p.relative_to(DST))

# ---------- (c) dump the pieces I need to read ----------
def dump(p: Path, title, limit=None):
    print("\n" + "=" * 78); print(title); print("=" * 78)
    if not p.exists():
        print("MISSING:", p); return
    txt = p.read_text()
    print(txt if limit is None else txt[:limit])

dump(DST / "pyproject.toml", "pyproject.toml  (dependency list)")
dump(DST / "src" / "tracking_cellmot" / "metrics.py", "metrics.py  ***THE SCORER***")
dump(DST / "scripts" / "evaluate.py", "scripts/evaluate.py")

# io.py signatures only (keep output short)
iop = DST / "src" / "tracking_cellmot" / "io.py"
if iop.exists():
    print("\n" + "=" * 78); print("io.py  (def lines only)"); print("=" * 78)
    for i, line in enumerate(iop.read_text().splitlines()):
        s = line.strip()
        if s.startswith(("def ", "class ", "@dataclass")):
            print(f"{i:5d}: {line}")

print("\nSTEP 3 COMPLETE")

PILKWANG PACK CONTENTS

[biohub-deepcenter-unet3d-center-prior-v1]
    ARTIFACT_MANIFEST.json  (0.0 MB)
    README.md  (0.0 MB)
    DIR source_scripts/
    source_scripts/build_full_frame_center_pack.py  (0.0 MB)
    source_scripts/package_and_upload_full_frame_center_pack.sh  (0.0 MB)
    source_scripts/run_full_frame_center_training.sh  (0.0 MB)
    source_scripts/train_full_frame_center_detector.py  (0.0 MB)
    DIR weights/
    DIR weights/full_frame_center/
    weights/full_frame_center/SNAPSHOT_MANIFEST.json  (0.0 MB)
    weights/full_frame_center/best.pt  (37.9 MB)
    weights/full_frame_center/checkpoint_last.pt  (37.9 MB)
    weights/full_frame_center/config.json  (0.0 MB)
    weights/full_frame_center/gate_frame_metrics.csv  (0.2 MB)
    weights/full_frame_center/gate_peak_samples.csv  (3.4 MB)
    weights/full_frame_center/gate_summary.json  (0.0 MB)
    weights/full_frame_center/gate_threshold_metrics.csv  (0.0 MB)
    weights/full_frame_center/history.csv  (0.0 MB)
    wei

In [12]:
# ==========================================================================
# STEP 4 — NATIVE SCORER + METRIC RESPONSE SURFACE (no tracksdata needed)
# Uses scipy bipartite matching. CPU, ~3 min.
# ==========================================================================
import json, warnings, sys
import numpy as np
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from collections import defaultdict

COMP  = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
TRAIN = COMP / "train"
SCALE = np.array([1.625, 0.40625, 0.40625])   # z, y, x µm/voxel
MAX_DIST_UM = 7.0
ALPHA       = 0.1

# ── zarr-v3 reader (same as Step 2) ──────────────────────────────────────
import zstandard as _zstd
_ZD = _zstd.ZstdDecompressor()

def _zdec(b): return _ZD.decompressobj().decompress(b)

def _read_v3(path: Path) -> np.ndarray:
    meta  = json.loads((path / "zarr.json").read_text())
    shape = tuple(int(s) for s in meta["shape"])
    dt    = np.dtype(meta["data_type"])
    chunk = tuple(int(c) for c in meta["chunk_grid"]["configuration"]["chunk_shape"])
    sep   = meta.get("chunk_key_encoding",{}).get("configuration",{}).get("separator","/")
    little= True
    for c in meta.get("codecs",[]):
        if c.get("name") == "bytes":
            little = c.get("configuration",{}).get("endian","little") == "little"
    dt = dt.newbyteorder("<" if little else ">")
    out = np.zeros(shape, dtype=dt)
    for idx in np.ndindex(*[int(np.ceil(shape[i]/chunk[i])) for i in range(len(shape))]):
        key = path / ("c" + sep + sep.join(str(i) for i in idx))
        sl  = tuple(slice(idx[i]*chunk[i], min((idx[i]+1)*chunk[i],shape[i]))
                    for i in range(len(shape)))
        if not key.exists(): continue
        arr = np.frombuffer(_zdec(key.read_bytes()), dtype=dt)
        want = tuple(s.stop-s.start for s in sl)
        arr  = arr.reshape(chunk)[tuple(slice(0,w) for w in want)] if arr.size==int(np.prod(chunk)) else arr.reshape(want)
        out[sl] = arr
    return out

def load_geff(stem: str):
    p   = TRAIN / f"{stem}.geff"
    ids = _read_v3(p / "nodes" / "ids")
    t   = _read_v3(p / "nodes" / "props" / "t" / "values")
    z   = _read_v3(p / "nodes" / "props" / "z" / "values")
    y   = _read_v3(p / "nodes" / "props" / "y" / "values")
    x   = _read_v3(p / "nodes" / "props" / "x" / "values")
    e   = _read_v3(p / "edges" / "ids")
    return ids.astype(np.int64), t.astype(np.int32), z.astype(np.float64), \
           y.astype(np.float64), x.astype(np.float64), e.astype(np.int64)

def est_total(stem: str) -> float:
    m = json.loads((TRAIN / f"{stem}.geff" / "zarr.json").read_text())
    st = [m]
    while st:
        c = st.pop()
        if isinstance(c, dict):
            if "estimated_number_of_nodes" in c:
                return float(c["estimated_number_of_nodes"])
            st.extend(c.values())
        elif isinstance(c, list): st.extend(c)
    return float("nan")

# ── native scorer ─────────────────────────────────────────────────────────
def match_nodes(pred_zyx_um: np.ndarray, gt_zyx_um: np.ndarray):
    """Optimal bipartite matching within MAX_DIST_UM.
    Returns pred_to_gt, gt_to_pred (index arrays, -1 = unmatched)."""
    N, M = len(pred_zyx_um), len(gt_zyx_um)
    if N == 0 or M == 0:
        return np.full(N, -1, np.int32), np.full(M, -1, np.int32)
    # distance matrix (µm)
    diff  = pred_zyx_um[:, None, :] - gt_zyx_um[None, :, :]
    dist  = np.sqrt((diff**2).sum(-1))           # (N, M)
    cost  = np.where(dist <= MAX_DIST_UM, dist, 1e9)
    ri, ci = linear_sum_assignment(cost)
    p2g = np.full(N, -1, np.int32)
    g2p = np.full(M, -1, np.int32)
    for r, c in zip(ri, ci):
        if cost[r, c] < 1e8:
            p2g[r] = c; g2p[c] = r
    return p2g, g2p

def score_sample(pred_pos4: np.ndarray, pred_edges: list,
                 gt_pos4: np.ndarray,   gt_edges: list,
                 et: float) -> dict:
    """
    pos4: (N,4) array of [t, z, y, x] in voxels.
    edges: list of (src_local_idx, tgt_local_idx).
    et: estimated true node count.
    """
    N = len(pred_pos4)
    # Physical positions for matching
    pred_um = pred_pos4[:, 1:] * SCALE   # (N,3) z,y,x µm
    gt_um   = gt_pos4[:,   1:] * SCALE

    p2g, g2p = match_nodes(pred_um, gt_um)

    # GT node out/in degrees
    gt_out = np.zeros(len(gt_pos4), np.int32)
    gt_in  = np.zeros(len(gt_pos4), np.int32)
    for a, b in gt_edges:
        gt_out[a] += 1; gt_in[b] += 1
    gt_edge_set = set(gt_edges)

    # Score predicted edges
    edge_tp = edge_fp = 0
    seen_pred = set()   # deduplicate
    seen_gt   = set()   # count each GT edge at most once as TP
    out_rank  = defaultdict(int)  # cap out-degree at 2

    pred_t = pred_pos4[:, 0].astype(int)
    gt_t   = gt_pos4[:,   0].astype(int)

    for (si, ti) in pred_edges:
        if si < 0 or ti < 0 or si >= N or ti >= N: continue
        # consecutive frames only
        if pred_t[ti] != pred_t[si] + 1: continue
        # out-degree cap (keep first 2 per source)
        out_rank[si] += 1
        if out_rank[si] > 2: continue
        # dedup same src→tgt
        key = (si, ti)
        if key in seen_pred: continue
        seen_pred.add(key)

        sg, tg = int(p2g[si]), int(p2g[ti])
        gt_key = (sg, tg)
        is_tp  = sg >= 0 and tg >= 0 and gt_key in gt_edge_set and gt_key not in seen_gt
        out_v  = sg >= 0 and gt_out[sg] > 0
        in_v   = tg >= 0 and gt_in[tg]  > 0

        if is_tp:
            edge_tp += 1; seen_gt.add(gt_key)
        elif out_v or in_v:
            edge_fp += 1

    edge_fn  = len(gt_edges) - edge_tp
    denom    = edge_tp + edge_fp + edge_fn
    jaccard  = edge_tp / denom if denom > 0 else 0.0
    ratio    = (N - et) / et  if et > 0    else 0.0
    adj      = max(0.0, jaccard * (1.0 - ALPHA * ratio))

    return dict(edge_tp=edge_tp, edge_fp=edge_fp, edge_fn=edge_fn,
                num_pred=N, jaccard=jaccard, adj=adj, ratio=ratio,
                multiplier=1.0-ALPHA*ratio)

def summarise_rows(rows):
    tp  = sum(r["edge_tp"]  for r in rows)
    fp  = sum(r["edge_fp"]  for r in rows)
    fn  = sum(r["edge_fn"]  for r in rows)
    J   = tp/(tp+fp+fn) if (tp+fp+fn) else 0.0
    # weight-averaged adj_J by sample w = tp+fp+fn
    ws  = [r["edge_tp"]+r["edge_fp"]+r["edge_fn"] for r in rows]
    W   = sum(ws)
    adjJ = sum(w*r["adj"] for w,r in zip(ws,rows))/W if W else 0.0
    # ignoring division term for this surface (tiny contribution, no GT divs in our 4 samples)
    return dict(edgeJ=J, adjJ=adjJ, score=adjJ)

# ── load four representative samples ─────────────────────────────────────
SAMPLES = ["6bba_05b6850b", "6bba_2540cd90", "44b6_341df25f", "44b6_95029e92"]

print("="*78); print("LOADING SAMPLES"); print("="*78)
gts = {}   # stem -> (pos4, edges, et)
for nm in SAMPLES:
    ids, t, z, y, x, e = load_geff(nm)
    pos4 = np.stack([t, z, y, x], 1).astype(np.float64)
    idx  = {int(v): i for i,v in enumerate(ids)}
    edges = [(idx[int(a)], idx[int(b)]) for a,b in zip(e[:,0], e[:,1])]
    et   = est_total(nm)
    gts[nm] = (pos4, edges, et)
    dens = 100*len(ids)/et
    mult = 1.0 - ALPHA*(len(ids)-et)/et
    print(f"  {nm:<22} nodes={len(ids):>5} edges={len(edges):>5} "
          f"est_true={et:>7.0f} density={dens:.2f}% theor_mult={mult:.4f}")

rng = np.random.default_rng(42)

def make_decoy_pos4(n, step_um=2.0):
    rows, edg = [], []
    base = 0
    sz = step_um / (SCALE * np.sqrt(3))
    while base < n:
        L  = min(int(rng.integers(10,30)), n-base)
        t0 = int(rng.integers(0, max(1, 100-L)))
        p  = rng.uniform([0,2,2],[63,253,253])
        for k in range(L):
            rows.append([t0+k, *np.clip(p,[0,0,0],[63,255,255])])
            if k > 0: edg.append((base+k-1, base+k))
            p += rng.normal(0,1,3)*sz
        base += L
    return np.array(rows, float), edg

# ── variant runner ────────────────────────────────────────────────────────
def run_variant(label, mutate):
    rows = []
    for nm in SAMPLES:
        pos4, edges, et = gts[nm]
        P, E = mutate(pos4.copy(), list(edges), et)
        r = score_sample(P, E, pos4, edges, et)
        rows.append(r)
    s = summarise_rows(rows)
    print(f"  {label:<40} score={s['score']:.4f}  "
          f"edgeJ={s['edgeJ']:.4f}  adjJ={s['adjJ']:.4f}")
    return s

# ── SMOKE TEST: GT scored against itself ─────────────────────────────────
print("\n"+"="*78); print("SMOKE TEST — GT vs GT (upper bound per sample)"); print("="*78)
for nm in SAMPLES:
    pos4, edges, et = gts[nm]
    r = score_sample(pos4, edges, pos4, edges, et)
    print(f"  {nm:<22} edgeJ={r['jaccard']:.4f}  adj={r['adj']:.4f}  "
          f"mult={r['multiplier']:.4f}  ratio={r['ratio']:.4f}")

# ── RESPONSE SURFACE ─────────────────────────────────────────────────────
print("\n"+"="*78); print("METRIC RESPONSE SURFACE"); print("="*78)

s_ceil = run_variant("A  perfect GT (ceiling)",
                     lambda p,e,et: (p, e))

print()
for mult in [0.1, 0.25, 0.5, 1.0, 2.0, 4.0]:
    def _add(p, e, et, m=mult):
        dp, de = make_decoy_pos4(int(m*et))
        off = len(p)
        return np.vstack([p,dp]), e+[(a+off,b+off) for a,b in de]
    run_variant(f"B  GT + {mult:.2f}x est_true decoy nodes+edges", _add)

print()
for frac in [0.10, 0.25, 0.50, 0.75]:
    def _drop(p, e, et, f=frac):
        keep = rng.random(len(p)) > f
        idx  = {o:i for i,o in enumerate(np.where(keep)[0])}
        return p[keep], [(idx[a],idx[b]) for a,b in e if a in idx and b in idx]
    run_variant(f"C  GT with {int(frac*100)}% nodes dropped", _drop)

print()
for frac in [0.05, 0.10, 0.25, 0.50]:
    def _swap(p, e, et, f=frac):
        e2 = list(e)
        k  = int(f*len(e2))
        for i in rng.choice(len(e2), min(k,len(e2)), replace=False):
            a,_ = e2[i]; e2[i]=(a, int(rng.integers(0,len(p))))
        return p, e2
    run_variant(f"D  {int(frac*100)}% edges misrouted", _swap)

print()
def _nodiv(p, e, et):
    seen,out = set(),[]
    for a,b in e:
        if a in seen: continue
        seen.add(a); out.append((a,b))
    return p, out
run_variant("E  GT divisions removed (all single-child)", _nodiv)

print()
# Extreme node pruning — can multiplier outrun recall loss?
for frac in [0.80, 0.90, 0.95, 0.99]:
    def _extreme(p, e, et, f=frac):
        keep = rng.random(len(p)) > f
        idx  = {o:i for i,o in enumerate(np.where(keep)[0])}
        return p[keep], [(idx[a],idx[b]) for a,b in e if a in idx and b in idx]
    run_variant(f"F  GT with {int(frac*100)}% nodes dropped (extreme)", _extreme)

print()
# Adding decoy nodes with NO extra edges — pure node penalty test
for m in [0.5, 1.0, 2.0, 4.0]:
    def _nodeonly(p, e, et, mul=m):
        dp, _ = make_decoy_pos4(int(mul*et))
        return np.vstack([p,dp]), e       # NO extra edges
    run_variant(f"G  perfect edges + {m:.1f}x isolated decoy nodes", _nodeonly)

# ── GRADIENT SUMMARY ─────────────────────────────────────────────────────
print("\n"+"="*78); print("GRADIENT SUMMARY"); print("="*78)
print(f"  Ceiling (GT-only prediction)    : {s_ceil['score']:.4f}")
print(f"  Your current submission         : ~0.913")
print(f"  Gap to ceiling on these samples : {s_ceil['score']-0.913:.4f}")
print()
print("  Key questions answered by this surface:")
print("  B-series: cost of over-detection (node penalty)")
print("  C-series: cost of missed detections (recall loss)")
print("  F-series: does extreme under-detection ever win?")
print("  G-series: do isolated extra nodes hurt if edges are perfect?")
print()
print("STEP 4 COMPLETE")

LOADING SAMPLES
  6bba_05b6850b          nodes=  861 edges=  845 est_true=   6362 density=13.53% theor_mult=1.0865
  6bba_2540cd90          nodes=  529 edges=  523 est_true=   3783 density=13.98% theor_mult=1.0860
  44b6_341df25f          nodes=  216 edges=  209 est_true=   7436 density=2.90% theor_mult=1.0971
  44b6_95029e92          nodes=  200 edges=  198 est_true=   5161 density=3.88% theor_mult=1.0961

SMOKE TEST — GT vs GT (upper bound per sample)
  6bba_05b6850b          edgeJ=1.0000  adj=1.0865  mult=1.0865  ratio=-0.8647
  6bba_2540cd90          edgeJ=1.0000  adj=1.0860  mult=1.0860  ratio=-0.8602
  44b6_341df25f          edgeJ=1.0000  adj=1.0971  mult=1.0971  ratio=-0.9710
  44b6_95029e92          edgeJ=1.0000  adj=1.0961  mult=1.0961  ratio=-0.9612

METRIC RESPONSE SURFACE
  A  perfect GT (ceiling)                  score=1.0887  edgeJ=1.0000  adjJ=1.0887

  B  GT + 0.10x est_true decoy nodes+edges score=1.0787  edgeJ=1.0000  adjJ=1.0787
  B  GT + 0.25x est_true decoy nodes+e

In [13]:
# ==========================================================================
# STEP 5 — VALIDATION HARNESS
# Installs from pilkwang wheels, runs inference on 8 training samples,
# scores with our native scorer. First real measurement on known GT.
# ~12 minutes on 2×T4.
# ==========================================================================
import os, sys, json, subprocess, shutil, warnings, time
import numpy as np
from pathlib import Path

COMP     = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
TRAIN    = COMP / "train"
WORK     = Path("/kaggle/working/val_run")
WORK.mkdir(exist_ok=True)
SCALE    = np.array([1.625, 0.40625, 0.40625])

# ── pick 8 validation samples, 4 per embryo, diverse densities ──────────
VAL_STEMS = [
    # 6bba — high annotation density, model was trained on full set so
    # this is contaminated but gives a relative baseline
    "6bba_05b6850b",    # est=6362,  density=13.5%
    "6bba_2540cd90",    # est=3783,  density=14.0%
    "6bba_09961292",    # est=31117, density=6.3%
    "6bba_57b7cc1e",    # est=65511, density=2.5%
    # 44b6 — lower density
    "44b6_341df25f",    # est=7436,  density=2.9%
    "44b6_95029e92",    # est=5161,  density=3.9%
    "44b6_d29c9ab2",    # est=38055, density=3.6%
    "44b6_3a861e03",    # est=40546, density=2.3%
]
print("Validation samples:", VAL_STEMS)

# ── Step 5a: install wheels (no internet needed) ─────────────────────────
WHEELS = Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/wheels")
PACK50 = Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1")

print("\n── installing from wheels ──")
needed = ["zarr", "geff", "numcodecs", "blosc2", "imagecodecs",
          "pyscipopt", "ilpy", "rustworkx", "tracksdata", "donfig",
          "bidict", "psygnal"]
whl_files = list(WHEELS.glob("*.whl"))
names_avail = [w.name for w in whl_files]

to_install = []
for pkg in needed:
    hit = [w for w in whl_files if w.name.lower().startswith(pkg.lower())]
    if hit:
        to_install.append(str(hit[0]))
    else:
        to_install.append(pkg)   # fall back to name (pip will resolve)

r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps"] + to_install,
    capture_output=True, text=True, timeout=600)
print("wheel install rc:", r.returncode)
if r.returncode != 0:
    print(r.stderr[-2000:])

# ── Step 5b: materialise the inference repo ──────────────────────────────
REPO = WORK / "repo"
if not REPO.exists():
    shutil.copytree(PACK50 / "repo", REPO)
    # also copy weights
    shutil.copytree(PACK50 / "weights", REPO / "weights")
    print("repo + weights copied")
else:
    print("repo already present")

# copy secondary weights
SEC_SRC = Path("/kaggle/input/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1")
SEC_DST = WORK / "secondary_weights"
if not SEC_DST.exists():
    shutil.copytree(SEC_SRC / "weights", SEC_DST)
    print("secondary weights copied")

# ── Step 5c: write a mini data dir with only our 8 samples ───────────────
VAL_DATA = WORK / "val_data"
VAL_DATA.mkdir(exist_ok=True)
for s in VAL_STEMS:
    dst_z = VAL_DATA / f"{s}.zarr"
    dst_g = VAL_DATA / f"{s}.geff"
    if not dst_z.exists():
        os.symlink(TRAIN / f"{s}.zarr", dst_z)
    if not dst_g.exists():
        os.symlink(TRAIN / f"{s}.geff", dst_g)
print(f"symlinked {len(VAL_STEMS)} samples into {VAL_DATA}")

# ── Step 5d: write splits file ────────────────────────────────────────────
splits_path = REPO / "val_splits.json"
splits_path.write_text(json.dumps([{"split": 0, "train": [], "test": VAL_STEMS}]))

# ── Step 5e: set env vars matching the submission notebook ────────────────
env = {**os.environ,
    "PYTHONPATH": str(REPO / "src"),
    # detection / ensemble
    "BIOHUB_DET_THRESHOLD":              "0.96875",
    "BIOHUB_SECONDARY_WEIGHTS":          str(SEC_DST / "unet_transformer/split_0/edge_predictor_best.pth"),
    "BIOHUB_SECONDARY_EDGE_WEIGHT":      "0.15",
    "BIOHUB_SECONDARY_DETECTION_WEIGHT": "0.475",
    "BIOHUB_SECONDARY_LINK_MODE":        "low_margin_consensus",
    "BIOHUB_SECONDARY_LOW_MARGIN_MAX":   "0.35",
    "BIOHUB_DUAL_SEED_EDGE_THRESHOLD":   "0.48",
    "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION": "0.90",
    # ILP
    "BIOHUB_ILP_APPEARANCE_WEIGHT":      "0.0",
    "BIOHUB_ILP_DISAPPEARANCE_WEIGHT":   "1.5",
    "BIOHUB_ILP_DIVISION_WEIGHT":        "1.0",
    # postprocessing
    "BIOHUB_OUTPUT_FILTER_SHORT_TRACKS": "1",
    "BIOHUB_OUTPUT_MIN_TRACK_LEN":       "6",
    "BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS": "1",
    "BIOHUB_GAP_CLOSE_MAX_GAP":          "2",
    "BIOHUB_GAP_CLOSE_UM":               "5.8",
    "BIOHUB_GAP_DENSITY_ADAPTIVE":       "1",
    "BIOHUB_SAFE_DIV_MAX_UM":            "4.66",
    "BIOHUB_SAFE_DIV_SISTER_MAX_UM":     "8.5",
}

# ── Step 5f: run prediction ───────────────────────────────────────────────
import torch
n_gpu = torch.cuda.device_count()
print(f"\n── running inference on {len(VAL_STEMS)} samples with {n_gpu} GPU(s) ──")

cmd = [
    sys.executable,
    str(REPO / "scripts" / "predict_unet_transformer.py"),
    "--data-dir", str(VAL_DATA),
    "--splits",   "val_splits.json",
    "--split",    "0",
    "--weights",  "weights/unet_transformer/split_0/edge_predictor_best.pth",
    "--unet-batch-size", "4",
    "--det-threshold",   "0.96875",
    "--ilp-edge-weight",         "-1.0",
    "--ilp-appearance-weight",   "0.0",
    "--ilp-disappearance-weight","1.5",
    "--ilp-division-weight",     "1.0",
    "--use-ilp",
]

t0 = time.time()
result = subprocess.run(cmd, cwd=REPO, env=env,
                        capture_output=False, timeout=3600)
elapsed = time.time() - t0
print(f"\nInference done in {elapsed/60:.1f} min | rc={result.returncode}")
if result.returncode != 0:
    print("INFERENCE FAILED — paste any error above"); sys.exit(0)

# ── Step 5g: find predicted geffs ─────────────────────────────────────────
pred_geffs = sorted((REPO / "predictions").rglob("*.geff"))
print(f"\nFound {len(pred_geffs)} predicted .geff files:")
for g in pred_geffs:
    print(f"  {g.stem}  ({g.stat().st_size/1e6:.1f} MB)")

# ── Step 5h: score with our native scorer ─────────────────────────────────
import zstandard as _zstd
_ZD2 = _zstd.ZstdDecompressor()
def _zdec2(b): return _ZD2.decompressobj().decompress(b)

def _read_v3(path):
    meta  = json.loads((path/"zarr.json").read_text())
    shape = tuple(int(s) for s in meta["shape"])
    dt    = np.dtype(meta["data_type"])
    chunk = tuple(int(c) for c in meta["chunk_grid"]["configuration"]["chunk_shape"])
    sep   = meta.get("chunk_key_encoding",{}).get("configuration",{}).get("separator","/")
    little= True
    for c in meta.get("codecs",[]):
        if c.get("name")=="bytes":
            little = c.get("configuration",{}).get("endian","little")=="little"
    dt = dt.newbyteorder("<" if little else ">")
    out= np.zeros(shape,dtype=dt)
    for idx in np.ndindex(*[int(np.ceil(shape[i]/chunk[i])) for i in range(len(shape))]):
        key=path/("c"+sep+sep.join(str(i) for i in idx))
        sl =tuple(slice(idx[i]*chunk[i],min((idx[i]+1)*chunk[i],shape[i]))
                  for i in range(len(shape)))
        if not key.exists(): continue
        arr=np.frombuffer(_zdec2(key.read_bytes()),dtype=dt)
        want=tuple(s.stop-s.start for s in sl)
        arr=arr.reshape(chunk)[tuple(slice(0,w) for w in want)] if arr.size==int(np.prod(chunk)) else arr.reshape(want)
        out[sl]=arr
    return out

def load_geff_any(p):
    p=Path(p)
    ids=_read_v3(p/"nodes"/"ids")
    t  =_read_v3(p/"nodes"/"props"/"t"/"values")
    z  =_read_v3(p/"nodes"/"props"/"z"/"values")
    y  =_read_v3(p/"nodes"/"props"/"y"/"values")
    x  =_read_v3(p/"nodes"/"props"/"x"/"values")
    e  =_read_v3(p/"edges"/"ids")
    return (ids.astype(np.int64), t.astype(np.int32),
            z.astype(np.float64), y.astype(np.float64), x.astype(np.float64),
            e.astype(np.int64))

def score_pair(pred_path, gt_stem):
    ids_p,tp,zp,yp,xp,ep = load_geff_any(pred_path)
    ids_g,tg,zg,yg,xg,eg = load_geff_any(TRAIN/f"{gt_stem}.geff")
    et = est_total_stem(gt_stem)

    pred_pos = np.stack([tp,zp,yp,xp],1).astype(float)
    gt_pos   = np.stack([tg,zg,yg,xg],1).astype(float)
    rp = {int(v):i for i,v in enumerate(ids_p)}
    rg = {int(v):i for i,v in enumerate(ids_g)}
    pe = [(rp[int(a)],rp[int(b)]) for a,b in zip(ep[:,0],ep[:,1])]
    ge = [(rg[int(a)],rg[int(b)]) for a,b in zip(eg[:,0],eg[:,1])]

    from scipy.optimize import linear_sum_assignment
    N,M = len(pred_pos),len(gt_pos)
    pred_um = pred_pos[:,1:]*SCALE
    gt_um   = gt_pos[:,1:] *SCALE
    diff  = pred_um[:,None,:]-gt_um[None,:,:]
    dist  = np.sqrt((diff**2).sum(-1))
    cost  = np.where(dist<=7.0, dist, 1e9)
    ri,ci = linear_sum_assignment(cost)
    p2g=np.full(N,-1,np.int32); g2p=np.full(M,-1,np.int32)
    for r,c in zip(ri,ci):
        if cost[r,c]<1e8: p2g[r]=c; g2p[c]=r
    node_recall = (g2p>=0).sum()/M if M>0 else 0.0

    gt_out=np.zeros(M,np.int32); gt_in=np.zeros(M,np.int32)
    for a,b in ge: gt_out[a]+=1; gt_in[b]+=1
    gt_set = set(ge)

    pred_t = pred_pos[:,0].astype(int)
    gt_t   = gt_pos[:,0].astype(int)
    tp_e=fp_e=0; seen_p=set(); seen_g=set(); out_rank={}
    for (si,ti) in pe:
        if si<0 or ti<0 or si>=N or ti>=N: continue
        if pred_t[ti]!=pred_t[si]+1: continue
        out_rank[si]=out_rank.get(si,0)+1
        if out_rank[si]>2: continue
        key=(si,ti)
        if key in seen_p: continue
        seen_p.add(key)
        sg,tg_=int(p2g[si]),int(p2g[ti])
        gk=(sg,tg_)
        is_tp = sg>=0 and tg_>=0 and gk in gt_set and gk not in seen_g
        out_v  = sg>=0 and gt_out[sg]>0
        in_v   = tg_>=0 and gt_in[tg_]>0
        if is_tp: tp_e+=1; seen_g.add(gk)
        elif out_v or in_v: fp_e+=1
    fn_e=len(ge)-tp_e
    denom=tp_e+fp_e+fn_e
    jac = tp_e/denom if denom>0 else 0.0
    ratio=(N-et)/et if et>0 else 0.0
    adj=max(0.0, jac*(1-0.1*ratio))
    return dict(stem=gt_stem, n_pred=N, n_gt=M, et=et,
                tp=tp_e, fp=fp_e, fn=fn_e,
                jaccard=jac, adj=adj, ratio=ratio, mult=1-0.1*ratio,
                node_recall=node_recall)

def est_total_stem(stem):
    m=json.loads((TRAIN/f"{stem}.geff"/"zarr.json").read_text())
    st=[m]
    while st:
        c=st.pop()
        if isinstance(c,dict):
            if "estimated_number_of_nodes" in c: return float(c["estimated_number_of_nodes"])
            st.extend(c.values())
        elif isinstance(c,list): st.extend(c)
    return float("nan")

print("\n"+"="*90)
print("VALIDATION RESULTS — actual predictions vs GT")
print("="*90)
print(f"{'stem':<24} {'n_pred':>8} {'et':>8} {'ratio':>7} {'mult':>6} "
      f"{'nrecall':>8} {'edgeJ':>7} {'adjJ':>7} {'TP':>6} {'FP':>6} {'FN':>6}")

all_rows = []
for g in pred_geffs:
    stem = g.stem
    if stem not in VAL_STEMS: continue
    try:
        r = score_pair(g, stem)
        all_rows.append(r)
        print(f"{r['stem']:<24} {r['n_pred']:>8} {r['et']:>8.0f} {r['ratio']:>7.3f} "
              f"{r['mult']:>6.3f} {r['node_recall']:>8.3f} {r['jaccard']:>7.4f} "
              f"{r['adj']:>7.4f} {r['tp']:>6} {r['fp']:>6} {r['fn']:>6}")
    except Exception as ex:
        print(f"  FAIL {stem}: {ex}")

if all_rows:
    ws=[r['tp']+r['fp']+r['fn'] for r in all_rows]; W=sum(ws)
    adj_avg = sum(w*r['adj'] for w,r in zip(ws,all_rows))/W if W else 0
    tp_=sum(r['tp'] for r in all_rows)
    fp_=sum(r['fp'] for r in all_rows)
    fn_=sum(r['fn'] for r in all_rows)
    J  =tp_/(tp_+fp_+fn_) if (tp_+fp_+fn_) else 0
    nr =sum(r['node_recall'] for r in all_rows)/len(all_rows)
    print("-"*90)
    print(f"{'MICRO-AVERAGE':<24} {'':>8} {'':>8} {'':>7} {'':>6} "
          f"{nr:>8.3f} {J:>7.4f} {adj_avg:>7.4f} {tp_:>6} {fp_:>6} {fn_:>6}")
    print(f"\nEstimated leaderboard score (no divisions): {adj_avg:.4f}")
    print(f"Ceiling on these samples:                   {1.0887:.4f}")
    print(f"Gap:                                        {1.0887-adj_avg:.4f}")
    print(f"\nedge Jaccard = TP/(TP+FP+FN) = {tp_}/{tp_+fp_+fn_} = {J:.4f}")
    print(f"Node recall (fraction of GT nodes matched)= {nr:.3f}")

print("\nSTEP 5 COMPLETE")

Validation samples: ['6bba_05b6850b', '6bba_2540cd90', '6bba_09961292', '6bba_57b7cc1e', '44b6_341df25f', '44b6_95029e92', '44b6_d29c9ab2', '44b6_3a861e03']

── installing from wheels ──
wheel install rc: 0
repo + weights copied
secondary weights copied
symlinked 8 samples into /kaggle/working/val_run/val_data

── running inference on 8 samples with 2 GPU(s) ──


/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


Fold 0: 8 datasets | weights=weights/unet_transformer/split_0/edge_predictor_best.pth | device=cuda | window_size=2 | pool_kernel_um=3.0
Saved 8 predictions to /kaggle/working/val_run/repo/predictions/unknown/unet_transformer/split_0

Inference done in 11.3 min | rc=0

Found 8 predicted .geff files:
  44b6_341df25f  (0.0 MB)
  44b6_3a861e03  (0.0 MB)
  44b6_95029e92  (0.0 MB)
  44b6_d29c9ab2  (0.0 MB)
  6bba_05b6850b  (0.0 MB)
  6bba_09961292  (0.0 MB)
  6bba_2540cd90  (0.0 MB)
  6bba_57b7cc1e  (0.0 MB)

VALIDATION RESULTS — actual predictions vs GT
stem                       n_pred       et   ratio   mult  nrecall   edgeJ    adjJ     TP     FP     FN
44b6_341df25f                8560     7436   0.151  0.985    1.000  0.2204  0.2171     82    163    127
44b6_3a861e03               37814    40546  -0.067  1.007    1.000  0.1580  0.1590    290    917    629
44b6_95029e92                4856     5161  -0.059  1.006    1.000  0.2844  0.2861     91    122    107
44b6_d29c9ab2               

In [15]:
# ==========================================================================
# STEP 6 — POSTPROCESSING PARAMETER SWEEP on cached predictions (CPU only)
# We already have 8 predicted .geff from Step 5.
# No inference needed. Re-runs only the filter_output_graph equivalent.
# ==========================================================================
import json, sys, warnings, os
import numpy as np
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
import zstandard as _zstd

COMP  = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
TRAIN = COMP / "train"
REPO  = Path("/kaggle/working/val_run/repo")
PRED  = sorted((REPO / "predictions").rglob("*.geff"))
SCALE = np.array([1.625, 0.40625, 0.40625])
_ZD   = _zstd.ZstdDecompressor()

# ── zarr reader ────────────────────────────────────────────────────────────
def _zdec(b): return _ZD.decompressobj().decompress(b)

def _rv3(path):
    path = Path(path)
    meta  = json.loads((path/"zarr.json").read_text())
    shape = tuple(int(s) for s in meta["shape"])
    dt    = np.dtype(meta["data_type"])
    chunk = tuple(int(c) for c in meta["chunk_grid"]["configuration"]["chunk_shape"])
    sep   = meta.get("chunk_key_encoding",{}).get("configuration",{}).get("separator","/")
    little = True
    for c in meta.get("codecs",[]):
        if c.get("name")=="bytes":
            little=c.get("configuration",{}).get("endian","little")=="little"
    dt=dt.newbyteorder("<" if little else ">")
    out=np.zeros(shape,dtype=dt)
    for idx in np.ndindex(*[int(np.ceil(shape[i]/chunk[i])) for i in range(len(shape))]):
        key=path/("c"+sep+sep.join(str(i) for i in idx))
        sl=tuple(slice(idx[i]*chunk[i],min((idx[i]+1)*chunk[i],shape[i]))
                 for i in range(len(shape)))
        if not key.exists(): continue
        arr=np.frombuffer(_zdec(key.read_bytes()),dtype=dt)
        want=tuple(s.stop-s.start for s in sl)
        arr=arr.reshape(chunk)[tuple(slice(0,w) for w in want)] if arr.size==int(np.prod(chunk)) else arr.reshape(want)
        out[sl]=arr
    return out

def load_geff_raw(p):
    p=Path(p)
    ids=_rv3(p/"nodes"/"ids")
    t  =_rv3(p/"nodes"/"props"/"t"/"values")
    z  =_rv3(p/"nodes"/"props"/"z"/"values")
    y  =_rv3(p/"nodes"/"props"/"y"/"values")
    x  =_rv3(p/"nodes"/"props"/"x"/"values")
    e  =_rv3(p/"edges"/"ids")
    # load edge_prob if present
    ep_path = p/"edges"/"props"/"edge_prob"/"values"
    ep = _rv3(ep_path).astype(float) if ep_path.exists() else None
    return (ids.astype(np.int64), t.astype(np.int32),
            z.astype(np.float64), y.astype(np.float64), x.astype(np.float64),
            e.astype(np.int64), ep)

def est_total(stem):
    m=json.loads((TRAIN/f"{stem}.geff"/"zarr.json").read_text())
    st=[m]
    while st:
        c=st.pop()
        if isinstance(c,dict):
            if "estimated_number_of_nodes" in c: return float(c["estimated_number_of_nodes"])
            st.extend(c.values())
        elif isinstance(c,list): st.extend(c)
    return float("nan")

# ── native scorer ─────────────────────────────────────────────────────────
def score_arrays(pred_pos4, pred_edges, gt_pos4, gt_edges, et,
                 max_dist=7.0):
    N,M = len(pred_pos4), len(gt_pos4)
    pred_um = pred_pos4[:,1:]*SCALE
    gt_um   = gt_pos4[:,1:]  *SCALE
    # match nodes
    p2g = np.full(N,-1,np.int32); g2p=np.full(M,-1,np.int32)
    if N>0 and M>0:
        diff  = pred_um[:,None,:]-gt_um[None,:,:]
        dist  = np.sqrt((diff**2).sum(-1))
        cost  = np.where(dist<=max_dist,dist,1e9)
        ri,ci = linear_sum_assignment(cost)
        for r,c in zip(ri,ci):
            if cost[r,c]<1e8: p2g[r]=c; g2p[c]=r
    nr = (g2p>=0).sum()/M if M>0 else 0.0
    # gt degrees
    gt_out=np.zeros(M,np.int32); gt_in=np.zeros(M,np.int32)
    for a,b in gt_edges: gt_out[a]+=1; gt_in[b]+=1
    gt_set=set(gt_edges)
    # score edges
    pred_t=pred_pos4[:,0].astype(int)
    tp=fp=0; sp=set(); sg=set(); ork={}
    for si,ti in pred_edges:
        if si<0 or ti<0 or si>=N or ti>=N: continue
        if pred_t[ti]!=pred_t[si]+1: continue
        ork[si]=ork.get(si,0)+1
        if ork[si]>2: continue
        k=(si,ti)
        if k in sp: continue
        sp.add(k)
        pg,tg=int(p2g[si]),int(p2g[ti])
        gk=(pg,tg)
        is_tp=pg>=0 and tg>=0 and gk in gt_set and gk not in sg
        ov=pg>=0 and gt_out[pg]>0
        iv=tg>=0 and gt_in[tg]>0
        if is_tp: tp+=1; sg.add(gk)
        elif ov or iv: fp+=1
    fn=len(gt_edges)-tp
    denom=tp+fp+fn
    jac=tp/denom if denom>0 else 0.0
    ratio=(N-et)/et if et>0 else 0.0
    adj=max(0.0,jac*(1-0.1*ratio))
    return dict(tp=tp,fp=fp,fn=fn,N=N,jac=jac,adj=adj,
                ratio=ratio,mult=1-0.1*ratio,nr=nr)

def summarise(rows):
    tp=sum(r["tp"] for r in rows); fp=sum(r["fp"] for r in rows)
    fn=sum(r["fn"] for r in rows)
    ws=[r["tp"]+r["fp"]+r["fn"] for r in rows]; W=sum(ws)
    adjJ=sum(w*r["adj"] for w,r in zip(ws,rows))/W if W else 0
    J=tp/(tp+fp+fn) if (tp+fp+fn) else 0
    nr=sum(r["nr"] for r in rows)/len(rows)
    return dict(edgeJ=J,adjJ=adjJ,score=adjJ,tp=tp,fp=fp,fn=fn,nr=nr)

# ── load raw predictions + GT once ────────────────────────────────────────
print("Loading predictions + GT …")
samples = {}
for gp in PRED:
    stem = gp.stem
    ids_p,tp_,zp,yp,xp,ep_,eprob = load_geff_raw(gp)
    ids_g,tg_,zg,yg,xg,eg_,_     = load_geff_raw(TRAIN/f"{stem}.geff")
    pp4 = np.stack([tp_,zp,yp,xp],1).astype(float)
    gp4 = np.stack([tg_,zg,yg,xg],1).astype(float)
    rp  = {int(v):i for i,v in enumerate(ids_p)}
    rg  = {int(v):i for i,v in enumerate(ids_g)}
    pe  = [(rp[int(a)],rp[int(b)]) for a,b in zip(ep_[:,0],ep_[:,1])]
    ge  = [(rg[int(a)],rg[int(b)]) for a,b in zip(eg_[:,0],eg_[:,1])]
    et  = est_total(stem)
    # edge_prob per predicted edge (if available)
    ep_arr = eprob if eprob is not None else None
    samples[stem] = dict(pp4=pp4, gp4=gp4, pe=pe, ge=ge, et=et,
                         ep_arr=ep_arr, ids_p=ids_p)
    print(f"  {stem}: pred={len(pp4)} edges={len(pe)} gt_nodes={len(gp4)} gt_edges={len(ge)} et={et:.0f}")

# helper: apply postprocessing filters and score
def run_config(label, cfg):
    """
    cfg keys:
      min_track_len   : int   (filter components shorter than this)
      max_edge_um     : float (drop edges longer than this µm)
      drop_isolated   : bool  (remove nodes with no edges)
      edge_prob_thresh: float (drop edges with prob < threshold; None=keep all)
      max_dist_match  : float (matching radius for scoring)
    """
    min_len   = cfg.get("min_track_len", 6)
    max_um    = cfg.get("max_edge_um", 14.0)
    drop_iso  = cfg.get("drop_isolated", True)
    ep_thresh = cfg.get("edge_prob_thresh", None)
    match_r   = cfg.get("max_dist_match", 7.0)

    rows=[]
    for stem, s in samples.items():
        pp4=s["pp4"]; pe_all=list(s["pe"]); gp4=s["gp4"]
        ge=s["ge"];   et=s["et"];            ep_arr=s["ep_arr"]

        # 1. edge_prob threshold
        if ep_thresh is not None and ep_arr is not None:
            pe_all = [(a,b) for (a,b),pr in zip(pe_all,ep_arr)
                      if float(pr)>=ep_thresh]

        # 2. drop edges > max_um
        def dist_um(a,b):
            return float(np.linalg.norm((pp4[a,1:]-pp4[b,1:])*SCALE))
        pe_all = [(a,b) for a,b in pe_all if dist_um(a,b)<=max_um]

        # 3. short-track filter (union-find on remaining edges)
        if min_len>1:
            N=len(pp4)
            par=list(range(N))
            def find(i):
                while par[i]!=i: par[i]=par[par[i]]; i=par[i]
                return i
            for a,b in pe_all:
                ra,rb=find(a),find(b)
                if ra!=rb: par[ra]=rb
            from collections import Counter
            comp=Counter(find(i) for i in range(N))
            keep_roots={r for r,cnt in comp.items() if cnt>=min_len}
            keep_nodes={i for i in range(N) if find(i) in keep_roots}
            pe_all=[(a,b) for a,b in pe_all if a in keep_nodes and b in keep_nodes]
            pp4_filt=pp4[sorted(keep_nodes)]
            remap={old:new for new,old in enumerate(sorted(keep_nodes))}
            pe_all=[(remap[a],remap[b]) for a,b in pe_all]
        else:
            pp4_filt=pp4; remap=None

        # 4. drop isolated
        if drop_iso and len(pe_all)>0:
            incident={a for a,b in pe_all}|{b for a,b in pe_all}
            keep2=sorted(incident)
            if remap: pass  # already filtered
            pp4_filt=pp4_filt[[i for i in range(len(pp4_filt)) if i in incident]] \
                if remap is None else pp4_filt
            # just reindex
            r2={old:new for new,old in enumerate(sorted(incident))}
            pe_all=[(r2[a],r2[b]) for a,b in pe_all if a in r2 and b in r2]
            pp4_filt=pp4_filt[sorted(incident)]

        r=score_arrays(pp4_filt, pe_all, gp4, ge, et, max_dist=match_r)
        rows.append(r)

    s=summarise(rows)
    print(f"  {label:<48} score={s['score']:.4f}  "
          f"edgeJ={s['edgeJ']:.4f}  adjJ={s['adjJ']:.4f}  "
          f"TP={s['tp']}  FP={s['fp']}  FN={s['fn']}  recall={s['nr']:.3f}")
    return s

print("\n" + "="*90)
print("POSTPROCESSING SWEEP on 8 cached predictions")
print("="*90)

# baseline (matches submission notebook settings)
print("\n-- BASELINE --")
s0 = run_config("BASELINE (min_len=6, max_um=14)",
                dict(min_track_len=6, max_edge_um=14.0, drop_isolated=True))

# ── A: min_track_len sweep ────────────────────────────────────────────────
print("\n-- A: min_track_len sweep --")
for L in [1, 2, 3, 4, 5, 6, 8, 10]:
    run_config(f"min_track_len={L}", dict(min_track_len=L, max_edge_um=14.0))

# ── B: max_edge_um sweep ─────────────────────────────────────────────────
print("\n-- B: max_edge_um sweep --")
for um in [6.0, 8.0, 10.0, 12.0, 14.0, 18.0]:
    run_config(f"max_edge_um={um}", dict(min_track_len=6, max_edge_um=um))

# ── C: edge_prob_threshold sweep (if edge_prob stored in geff) ────────────
print("\n-- C: edge_prob_threshold sweep --")
has_ep = any(s["ep_arr"] is not None for s in samples.values())
if has_ep:
    for thr in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
        run_config(f"edge_prob_thresh={thr}", dict(min_track_len=6,edge_prob_thresh=thr))
else:
    print("  edge_prob not stored in predicted geff — skip")

# ── D: combined best settings ─────────────────────────────────────────────
print("\n-- D: combined candidates --")
run_config("min_len=3 + max_um=10",
           dict(min_track_len=3, max_edge_um=10.0))
run_config("min_len=4 + max_um=10",
           dict(min_track_len=4, max_edge_um=10.0))
run_config("min_len=2 + max_um=12",
           dict(min_track_len=2, max_edge_um=12.0))
run_config("min_len=1 (no filter) + max_um=14",
           dict(min_track_len=1, max_edge_um=14.0))
run_config("drop_isolated=False + min_len=6",
           dict(min_track_len=6, max_edge_um=14.0, drop_isolated=False))

# ── E: matching radius sensitivity ───────────────────────────────────────
print("\n-- E: GT matching radius (scoring artefact check) --")
for r in [5.0, 6.0, 7.0, 8.0]:
    run_config(f"match_radius={r}um",
               dict(min_track_len=6, max_edge_um=14.0, max_dist_match=r))

print("\n" + "="*90)
print("INTERPRETATION GUIDE")
print("="*90)
print("""
  Score increases when:
    - TP rises (more correct GT edges found)  → raise score
    - FP falls (fewer wrong edges on annotated cells) → raise score
    - FN falls (fewer missed GT edges) → raise score
    - adjJ bonus from fewer nodes → minor effect here

  If min_track_len=1 beats min_track_len=6:
      → SHORT TRACK FILTER is deleting real GT edges on training set
      → Lower it in the submission notebook

  If max_edge_um=8 beats max_edge_um=14:
      → Current gate is too loose; wrong long-range links are FP
      → Tighten in submission

  If edge_prob_thresh>0 beats 0:
      → Low-confidence edges are net harmful; threshold them
      → Add edge_prob gate to submission postprocessing
""")
print("STEP 6 COMPLETE")

Loading predictions + GT …
  44b6_341df25f: pred=8560 edges=8148 gt_nodes=216 gt_edges=209 et=7436
  44b6_3a861e03: pred=37814 edges=35780 gt_nodes=935 gt_edges=919 et=40546
  44b6_95029e92: pred=4856 edges=4684 gt_nodes=200 gt_edges=198 et=5161
  44b6_d29c9ab2: pred=36907 edges=35539 gt_nodes=1353 gt_edges=1328 et=38055
  6bba_05b6850b: pred=6362 edges=5985 gt_nodes=861 gt_edges=845 et=6362
  6bba_09961292: pred=29169 edges=27611 gt_nodes=1950 gt_edges=1871 et=31117
  6bba_2540cd90: pred=3470 edges=3385 gt_nodes=529 gt_edges=523 et=3783
  6bba_57b7cc1e: pred=76885 edges=68447 gt_nodes=1659 gt_edges=1592 et=65511

POSTPROCESSING SWEEP on 8 cached predictions

-- BASELINE --
  BASELINE (min_len=6, max_um=14)                  score=0.1739  edgeJ=0.1727  adjJ=0.1739  TP=2486  FP=6914  FN=4999  recall=0.999

-- A: min_track_len sweep --


IndexError: index 4855 is out of bounds for axis 0 with size 4855

In [16]:
# ==========================================================================
# STEP 6 (FIXED) — redefine run_config, then rerun sweep
# Add as a NEW cell below the broken one. Do not edit the broken cell.
# ==========================================================================
from collections import Counter

def run_config(label, cfg):
    min_len   = cfg.get("min_track_len", 6)
    max_um    = cfg.get("max_edge_um", 14.0)
    drop_iso  = cfg.get("drop_isolated", True)
    ep_thresh = cfg.get("edge_prob_thresh", None)
    match_r   = cfg.get("max_dist_match", 7.0)

    rows = []
    for stem, s in samples.items():
        pp4    = s["pp4"].copy()   # work on a copy every time
        pe     = list(s["pe"])
        gp4    = s["gp4"]
        ge     = s["ge"]
        et     = s["et"]
        ep_arr = s["ep_arr"]

        # 1. edge_prob threshold
        if ep_thresh is not None and ep_arr is not None:
            pe = [(a,b) for (a,b),pr in zip(pe, ep_arr) if float(pr) >= ep_thresh]

        # 2. drop edges longer than max_um
        pe = [(a,b) for a,b in pe
              if np.linalg.norm((pp4[a,1:] - pp4[b,1:]) * SCALE) <= max_um]

        # 3. short-track filter (union-find, then reindex)
        if min_len > 1 and len(pe) > 0:
            N   = len(pp4)
            par = list(range(N))
            def find(i):
                while par[i] != i:
                    par[i] = par[par[i]]; i = par[i]
                return i
            for a, b in pe:
                ra, rb = find(a), find(b)
                if ra != rb: par[ra] = rb
            comp        = Counter(find(i) for i in range(N))
            keep_roots  = {r for r, cnt in comp.items() if cnt >= min_len}
            keep_nodes  = sorted(i for i in range(N) if find(i) in keep_roots)
            remap       = {old: new for new, old in enumerate(keep_nodes)}
            pp4         = pp4[keep_nodes]                               # shrink
            pe          = [(remap[a], remap[b]) for a,b in pe          # reindex
                           if a in remap and b in remap]

        # 4. drop isolated nodes (single reindex, no double-index)
        if drop_iso and len(pe) > 0:
            incident = sorted({a for a,b in pe} | {b for a,b in pe})
            remap2   = {old: new for new, old in enumerate(incident)}
            pp4      = pp4[incident]                                    # shrink once
            pe       = [(remap2[a], remap2[b]) for a,b in pe]          # reindex

        r = score_arrays(pp4, pe, gp4, ge, et, max_dist=match_r)
        rows.append(r)

    s = summarise(rows)
    print(f"  {label:<48} score={s['score']:.4f}  "
          f"edgeJ={s['edgeJ']:.4f}  adjJ={s['adjJ']:.4f}  "
          f"TP={s['tp']}  FP={s['fp']}  FN={s['fn']}  recall={s['nr']:.3f}")
    return s

# ── rerun the full sweep ───────────────────────────────────────────────────
print("="*90)
print("POSTPROCESSING SWEEP (fixed run_config)")
print("="*90)

print("\n-- BASELINE --")
s0 = run_config("BASELINE (min_len=6, max_um=14)",
                dict(min_track_len=6, max_edge_um=14.0, drop_isolated=True))

print("\n-- A: min_track_len sweep --")
for L in [1, 2, 3, 4, 5, 6, 8, 10]:
    run_config(f"min_track_len={L}",
               dict(min_track_len=L, max_edge_um=14.0))

print("\n-- B: max_edge_um sweep --")
for um in [6.0, 8.0, 10.0, 12.0, 14.0, 18.0]:
    run_config(f"max_edge_um={um}",
               dict(min_track_len=6, max_edge_um=um))

print("\n-- C: edge_prob_threshold sweep --")
has_ep = any(s["ep_arr"] is not None for s in samples.values())
if has_ep:
    for thr in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
        run_config(f"edge_prob_thresh={thr}",
                   dict(min_track_len=6, edge_prob_thresh=thr))
else:
    print("  edge_prob not in geff — skip")

print("\n-- D: combined candidates --")
run_config("min_len=1  max_um=14 (no filters)",
           dict(min_track_len=1, max_edge_um=14.0))
run_config("min_len=2  max_um=14",
           dict(min_track_len=2, max_edge_um=14.0))
run_config("min_len=3  max_um=14",
           dict(min_track_len=3, max_edge_um=14.0))
run_config("min_len=3  max_um=10",
           dict(min_track_len=3, max_edge_um=10.0))
run_config("min_len=4  max_um=10",
           dict(min_track_len=4, max_edge_um=10.0))
run_config("min_len=6  drop_iso=False",
           dict(min_track_len=6, drop_isolated=False))

print("\n-- E: matching radius sensitivity --")
for r in [5.0, 6.0, 7.0, 8.0]:
    run_config(f"match_radius={r}um",
               dict(min_track_len=6, max_edge_um=14.0, max_dist_match=r))

print("\nSTEP 6 COMPLETE")

POSTPROCESSING SWEEP (fixed run_config)

-- BASELINE --
  BASELINE (min_len=6, max_um=14)                  score=0.1739  edgeJ=0.1727  adjJ=0.1739  TP=2486  FP=6914  FN=4999  recall=0.999

-- A: min_track_len sweep --
  min_track_len=1                                  score=0.1716  edgeJ=0.1713  adjJ=0.1716  TP=2464  FP=6902  FN=5021  recall=1.000
  min_track_len=2                                  score=0.1716  edgeJ=0.1713  adjJ=0.1716  TP=2464  FP=6902  FN=5021  recall=1.000
  min_track_len=3                                  score=0.1716  edgeJ=0.1713  adjJ=0.1716  TP=2464  FP=6902  FN=5021  recall=1.000
  min_track_len=4                                  score=0.1723  edgeJ=0.1718  adjJ=0.1723  TP=2472  FP=6904  FN=5013  recall=1.000
  min_track_len=5                                  score=0.1722  edgeJ=0.1713  adjJ=0.1722  TP=2468  FP=6919  FN=5017  recall=0.999
  min_track_len=6                                  score=0.1739  edgeJ=0.1727  adjJ=0.1739  TP=2486  FP=6914  FN=4999  rec

In [19]:
# ==========================================================================
# STEP 7 (FIXED-2) — locate notebook without scanning data dirs
# ==========================================================================
import json, copy
from pathlib import Path

# ── check only lightweight directories ────────────────────────────────────
SEARCH_DIRS = [
    Path("/kaggle/working"),
    Path("/kaggle/input/datasets/pilkwang"),
    Path("/kaggle/input/datasets"),
]

found = []
for d in SEARCH_DIRS:
    if d.exists():
        for p in d.rglob("*.ipynb"):
            found.append(p)

print("Notebooks found:")
for p in found:
    print(f"  {p}  ({p.stat().st_size/1e3:.0f} kB)")

# ── also check if the current notebook is accessible via __file__ ─────────
import os
print("\nCurrent working dir:", os.getcwd())
print("Contents of /kaggle/working:")
for p in sorted(Path("/kaggle/working").iterdir()):
    print(f"  {'DIR' if p.is_dir() else 'FILE':4s} {p.name}  ({p.stat().st_size/1e3:.0f} kB)")

Notebooks found:
  /kaggle/working/.virtual_documents/__notebook_source__.ipynb  (64 kB)

Current working dir: /kaggle/working
Contents of /kaggle/working:
  DIR  .virtual_documents  (4 kB)
  DIR  official_repo  (4 kB)
  DIR  val_run  (4 kB)


In [20]:
# ── once you know the path, set SRC here and run this block ──────────────
# CHANGE THIS LINE to whatever path was printed above for the largest .ipynb
SRC = Path("/kaggle/working/YOUR_NOTEBOOK_NAME.ipynb")   # <-- edit this
# e.g. SRC = Path("/kaggle/working/biohub.ipynb")
# e.g. SRC = Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/repo/some.ipynb")

assert SRC.exists(), f"Not found: {SRC}"
print(f"Using {SRC}  ({SRC.stat().st_size/1e3:.0f} kB)")

nb    = json.loads(SRC.read_text())
nb    = copy.deepcopy(nb)
cells = nb["cells"]

def src_of(c): return "".join(c["source"])

# ── show the exact lines we need to patch ─────────────────────────────────
print("\n── Cell containing BIOHUB_OUTPUT_MIN_TRACK_LEN ──")
for i, c in enumerate(cells):
    if c["cell_type"] == "code" and "BIOHUB_OUTPUT_MIN_TRACK_LEN" in src_of(c):
        print(f"Cell index: {i}")
        for ln in src_of(c).splitlines():
            if any(k in ln for k in ["TRACK_LEN", "EDGE_PROB", "MIN_TRACK"]):
                print(f"  {repr(ln)}")
        break

print("\n── Cell containing filter_output_graph edge-append block ──")
for i, c in enumerate(cells):
    if c["cell_type"] == "code" and "filter_output_graph" in src_of(c):
        print(f"Cell index: {i}")
        lines = src_of(c).splitlines()
        for j, ln in enumerate(lines):
            if "edges.append" in ln or "OUTPUT_EDGE_MAX_UM" in ln or "dropped_long" in ln:
                # print 3 lines of context around each hit
                for k in range(max(0,j-1), min(len(lines), j+3)):
                    print(f"  {repr(lines[k])}")
                print("  ---")
        break

print("\nSTEP 7 locate COMPLETE — paste output and I will give the final patch cell")

AssertionError: Not found: /kaggle/working/YOUR_NOTEBOOK_NAME.ipynb